In [1]:
import mysql.connector
import pandas as pd
import numpy as np
from datetime import datetime

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="0987654321",
    database="datamart_rappi"
)
print("Conexión exitosa a datamart_rappi")

Conexión exitosa a datamart_rappi


In [3]:
# CELDA 2 --- BRONZE: Extraer todas las tablas

# Dimensiones
df_dim_tiempo = pd.read_sql("SELECT * FROM dim_tiempo", conn)
df_dim_zona = pd.read_sql("SELECT * FROM dim_zona", conn)
df_dim_categoria_comida = pd.read_sql("SELECT * FROM dim_categoria_comida", conn)
df_dim_cadena_restaurante = pd.read_sql("SELECT * FROM dim_cadena_restaurante", conn)
df_dim_restaurante = pd.read_sql("SELECT * FROM dim_restaurante", conn)
df_dim_tipo_vehiculo = pd.read_sql("SELECT * FROM dim_tipo_vehiculo", conn)
df_dim_repartidor = pd.read_sql("SELECT * FROM dim_repartidor", conn)
df_dim_tipo_suscripcion = pd.read_sql("SELECT * FROM dim_tipo_suscripcion", conn)
df_dim_metodo_pago = pd.read_sql("SELECT * FROM dim_metodo_pago", conn)
df_dim_cliente = pd.read_sql("SELECT * FROM dim_cliente", conn)
df_dim_estado_pedido = pd.read_sql("SELECT * FROM dim_estado_pedido", conn)

# Tabla de hechos
df_fact_pedidos = pd.read_sql("SELECT * FROM fact_pedidos", conn)

print("BRONZE — Tablas extraídas:")
print(f"   fact_pedidos:            {len(df_fact_pedidos):,} registros")
print(f"   dim_tiempo:              {len(df_dim_tiempo):,} registros")
print(f"   dim_zona:                {len(df_dim_zona):,} registros")
print(f"   dim_categoria_comida:    {len(df_dim_categoria_comida):,} registros")
print(f"   dim_cadena_restaurante:  {len(df_dim_cadena_restaurante):,} registros")
print(f"   dim_restaurante:         {len(df_dim_restaurante):,} registros")
print(f"   dim_tipo_vehiculo:       {len(df_dim_tipo_vehiculo):,} registros")
print(f"   dim_repartidor:          {len(df_dim_repartidor):,} registros")
print(f"   dim_tipo_suscripcion:    {len(df_dim_tipo_suscripcion):,} registros")
print(f"   dim_metodo_pago:         {len(df_dim_metodo_pago):,} registros")
print(f"   dim_cliente:             {len(df_dim_cliente):,} registros")
print(f"   dim_estado_pedido:       {len(df_dim_estado_pedido):,} registros")

C:\Users\Admin\AppData\Local\Temp\ipykernel_13352\158118650.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dim_tiempo = pd.read_sql("SELECT * FROM dim_tiempo", conn)
C:\Users\Admin\AppData\Local\Temp\ipykernel_13352\158118650.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dim_zona = pd.read_sql("SELECT * FROM dim_zona", conn)
C:\Users\Admin\AppData\Local\Temp\ipykernel_13352\158118650.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dim_categoria_comida = pd.read_sql("SELECT * FROM dim_categoria_comida

BRONZE — Tablas extraídas:
   fact_pedidos:            10,000 registros
   dim_tiempo:              5,988 registros
   dim_zona:                5 registros
   dim_categoria_comida:    5 registros
   dim_cadena_restaurante:  10 registros
   dim_restaurante:         10 registros
   dim_tipo_vehiculo:       3 registros
   dim_repartidor:          200 registros
   dim_tipo_suscripcion:    3 registros
   dim_metodo_pago:         5 registros
   dim_cliente:             3,000 registros
   dim_estado_pedido:       5 registros


In [5]:
# CELDA 3: BRONZE: Diagnóstico de calidad
print("=" * 60)
print("🔍 DIAGNÓSTICO BRONZE — CALIDAD DE DATOS")
print("=" * 60)

tablas = {
    'fact_pedidos': df_fact_pedidos,
    'dim_tiempo': df_dim_tiempo,
    'dim_zona': df_dim_zona,
    'dim_restaurante': df_dim_restaurante,
    'dim_repartidor': df_dim_repartidor,
    'dim_cliente': df_dim_cliente,
    'dim_estado_pedido': df_dim_estado_pedido
}

for nombre, df in tablas.items():
    nulos = df.isnull().sum().sum()
    duplicados = df.duplicated().sum()
    print(f"\n{nombre}:")
    print(f"   Registros: {len(df):,}")
    print(f"   Nulos totales: {nulos}")
    print(f"   Duplicados: {duplicados}")

🔍 DIAGNÓSTICO BRONZE — CALIDAD DE DATOS

fact_pedidos:
   Registros: 10,000
   Nulos totales: 0
   Duplicados: 0

dim_tiempo:
   Registros: 5,988
   Nulos totales: 0
   Duplicados: 0

dim_zona:
   Registros: 5
   Nulos totales: 0
   Duplicados: 0

dim_restaurante:
   Registros: 10
   Nulos totales: 0
   Duplicados: 0

dim_repartidor:
   Registros: 200
   Nulos totales: 0
   Duplicados: 0

dim_cliente:
   Registros: 3,000
   Nulos totales: 0
   Duplicados: 0

dim_estado_pedido:
   Registros: 5
   Nulos totales: 0
   Duplicados: 0


In [6]:
# CELDA 4 --- SILVER: Limpieza de dimensiones
print("=" * 60)
print("CAPA SILVER — LIMPIEZA DE DIMENSIONES")
print("=" * 60)

# Limpiar cada dimensión
def limpiar_tabla(df, nombre):
    antes = len(df)
    df = df.drop_duplicates()
    df = df.dropna()
    despues = len(df)
    print(f"{nombre}: {antes:,} → {despues:,} registros")
    return df

df_dim_tiempo_s = limpiar_tabla(df_dim_tiempo, 'dim_tiempo')
df_dim_zona_s = limpiar_tabla(df_dim_zona, 'dim_zona')
df_dim_categoria_comida_s = limpiar_tabla(df_dim_categoria_comida, 'dim_categoria_comida')
df_dim_cadena_restaurante_s = limpiar_tabla(df_dim_cadena_restaurante, 'dim_cadena_restaurante')
df_dim_restaurante_s = limpiar_tabla(df_dim_restaurante, 'dim_restaurante')
df_dim_tipo_vehiculo_s = limpiar_tabla(df_dim_tipo_vehiculo, 'dim_tipo_vehiculo')
df_dim_repartidor_s = limpiar_tabla(df_dim_repartidor, 'dim_repartidor')
df_dim_tipo_suscripcion_s = limpiar_tabla(df_dim_tipo_suscripcion, 'dim_tipo_suscripcion')
df_dim_metodo_pago_s = limpiar_tabla(df_dim_metodo_pago, 'dim_metodo_pago')
df_dim_cliente_s = limpiar_tabla(df_dim_cliente, 'dim_cliente')
df_dim_estado_pedido_s = limpiar_tabla(df_dim_estado_pedido, 'dim_estado_pedido')

CAPA SILVER — LIMPIEZA DE DIMENSIONES
dim_tiempo: 5,988 → 5,988 registros
dim_zona: 5 → 5 registros
dim_categoria_comida: 5 → 5 registros
dim_cadena_restaurante: 10 → 10 registros
dim_restaurante: 10 → 10 registros
dim_tipo_vehiculo: 3 → 3 registros
dim_repartidor: 200 → 200 registros
dim_tipo_suscripcion: 3 → 3 registros
dim_metodo_pago: 5 → 5 registros
dim_cliente: 3,000 → 3,000 registros
dim_estado_pedido: 5 → 5 registros


In [7]:
# CELDA 5 --- SILVER: Limpieza de fact_pedidos
print("=" * 60)
print("SILVER — LIMPIEZA FACT_PEDIDOS")
print("=" * 60)

df_fact_s = df_fact_pedidos.copy()
antes = len(df_fact_s)

# 1. Eliminar duplicados
df_fact_s = df_fact_s.drop_duplicates()
print(f"Duplicados eliminados: {antes - len(df_fact_s)}")

# 2. Eliminar nulos
df_fact_s = df_fact_s.dropna()
print(f"Nulos eliminados: {antes - len(df_fact_s)}")

# 3. Filtrar valores inválidos
df_fact_s = df_fact_s[df_fact_s['tiempo_total_min'] > 0]
df_fact_s = df_fact_s[df_fact_s['dist_asignacion_km'] >= 0]
df_fact_s = df_fact_s[df_fact_s['tiempo_espera_rest_min'] >= 0]
print(f"Valores inválidos eliminados")

print(f"\nfact_pedidos Silver: {len(df_fact_s):,} registros")

SILVER — LIMPIEZA FACT_PEDIDOS
Duplicados eliminados: 0
Nulos eliminados: 0
Valores inválidos eliminados

fact_pedidos Silver: 10,000 registros


In [8]:
# CELDA 6 --- SILVER: Transformaciones y métricas derivadas
print("=" * 60)
print("SILVER — TRANSFORMACIONES")
print("=" * 60)

# 1. Calcular minutos de retraso
SLA_OBJETIVO = 30
df_fact_s['minutos_retraso'] = df_fact_s['tiempo_total_min'].apply(
    lambda x: max(0, x - SLA_OBJETIVO)
)

# 2. Clasificar franja horaria
def clasificar_franja(hora):
    if 6 <= hora < 12:
        return 'Mañana'
    elif 12 <= hora < 18:
        return 'Tarde'
    elif 18 <= hora < 24:
        return 'Noche'
    else:
        return 'Madrugada'

df_dim_tiempo_s['franja_horaria'] = df_dim_tiempo_s['hora_dia'].apply(
    clasificar_franja
)

# 3. Categorizar distancia de asignación
def categorizar_distancia(dist):
    if dist < 1:
        return 'Corta (<1km)'
    elif dist < 2:
        return 'Media (1-2km)'
    else:
        return 'Larga (>2km)'

df_fact_s['categoria_distancia'] = df_fact_s['dist_asignacion_km'].apply(
    categorizar_distancia
)

# 4. Tiempo en tránsito
df_fact_s['tiempo_transito_min'] = (
    df_fact_s['tiempo_total_min'] - df_fact_s['tiempo_espera_rest_min']
)

print("Transformaciones aplicadas:")
print(f"   - minutos_retraso calculado (SLA objetivo: {SLA_OBJETIVO} min)")
print(f"   - franja_horaria clasificada en dim_tiempo")
print(f"   - categoria_distancia categorizada")
print(f"   - tiempo_transito_min calculado")
print(f"\nDistribución franja horaria:")
print(df_dim_tiempo_s['franja_horaria'].value_counts())
print(f"\nDistribución categoría distancia:")
print(df_fact_s['categoria_distancia'].value_counts())

SILVER — TRANSFORMACIONES
Transformaciones aplicadas:
   - minutos_retraso calculado (SLA objetivo: 30 min)
   - franja_horaria clasificada en dim_tiempo
   - categoria_distancia categorizada
   - tiempo_transito_min calculado

Distribución franja horaria:
franja_horaria
Noche        1517
Tarde        1505
Madrugada    1489
Mañana       1477
Name: count, dtype: int64

Distribución categoría distancia:
categoria_distancia
Media (1-2km)    5494
Corta (<1km)     2825
Larga (>2km)     1681
Name: count, dtype: int64


In [21]:
# CELDA 7 -- GOLD: Construir tabla analítica completa
print("=" * 60)
print("CAPA GOLD — MODELO DIMENSIONAL COMPLETO")
print("=" * 60)

# JOIN completo: fact_pedidos + todas las dimensiones
df_gold = df_fact_s.merge(
    df_dim_tiempo_s[['id_tiempo', 'hora_dia', 'dia_semana', 'mes', 'año', 
                      'es_hora_pico', 'franja_horaria']], 
    on='id_tiempo', how='left'
).merge(
    df_dim_zona_s[['id_zona', 'distrito', 'cuadrante_mapa', 'avenida_principal']], 
    on='id_zona', how='left'
).merge(
    df_dim_restaurante_s[['id_restaurante', 'nombre_sucursal', 
                           'calificacion_promedio', 'id_cadena']], 
    on='id_restaurante', how='left'
).merge(
    df_dim_cadena_restaurante_s[['id_cadena', 'nombre_cadena', 
                                  'tiene_sucursal_vip', 'id_categoria']], 
    on='id_cadena', how='left'
).merge(
    df_dim_repartidor_s[['id_repartidor', 'modalidad_reparto', 
                          'calificacion_repartidor', 'id_tipo_vehiculo']], 
    on='id_repartidor', how='left'
).merge(
    df_dim_tipo_vehiculo_s[['id_tipo_vehiculo', 'descripcion']], 
    on='id_tipo_vehiculo', how='left'
).merge(
    df_dim_estado_pedido_s[['id_estado', 'estado_actual', 'hubo_incidencia']], 
    on='id_estado', how='left'
)

print(f"GOLD: {len(df_gold):,} registros")
print(f"   Columnas disponibles: {len(df_gold.columns)}")
print(f"\nColumnas: {df_gold.columns.tolist()}")

CAPA GOLD — MODELO DIMENSIONAL COMPLETO
GOLD: 10,000 registros
   Columnas disponibles: 37

Columnas: ['id_pedido', 'id_tiempo', 'id_zona', 'id_restaurante', 'id_repartidor', 'id_cliente', 'id_estado', 'tiempo_total_min', 'dist_asignacion_km', 'dist_entrega_km', 'tiempo_espera_rest_min', 'cumple_sla', 'minutos_retraso', 'valor_pedido_soles', 'categoria_distancia', 'tiempo_transito_min', 'hora_dia', 'dia_semana', 'mes', 'año', 'es_hora_pico', 'franja_horaria', 'distrito', 'cuadrante_mapa', 'avenida_principal', 'nombre_sucursal', 'calificacion_promedio', 'id_cadena', 'nombre_cadena', 'tiene_sucursal_vip', 'id_categoria', 'modalidad_reparto', 'calificacion_repartidor', 'id_tipo_vehiculo', 'descripcion', 'estado_actual', 'hubo_incidencia']


In [ ]:
#print(df_dim_repartidor_s.columns.tolist())

['id_repartidor', 'modalidad_reparto', 'calificacion_repartidor', 'id_tipo_vehiculo']


In [19]:
# Data Cleaning: renombrar columnas con nombres incorrectos
df_dim_tiempo_s = df_dim_tiempo_s.rename(columns={
    'anio': 'año'
})

print("Data Cleaning aplicado:")
print("   - 'anio' renombrado a 'año' en dim_tiempo")
print(f"   Columnas corregidas: {df_dim_tiempo_s.columns.tolist()}")

Data Cleaning aplicado:
   - 'anio' renombrado a 'año' en dim_tiempo
   Columnas corregidas: ['id_tiempo', 'fecha', 'hora_dia', 'franja', 'dia_semana', 'mes', 'año', 'es_hora_pico', 'franja_horaria']


In [22]:
# CELDA 8 --- GOLD: KPIs finales

print("=" * 60)
print("KPIs FINALES — CAPA GOLD")
print("=" * 60)

kpi1 = df_gold['cumple_sla'].mean() * 100
kpi2 = df_gold['minutos_retraso'].mean()
kpi3 = df_gold['dist_asignacion_km'].mean()
kpi4 = df_gold['tiempo_espera_rest_min'].mean()
kpi5 = df_gold['tiempo_total_min'].mean()

print(f"\nKPI 1 — Cumplimiento SLA:           {kpi1:.2f}%")
print(f"KPI 2 — Retraso Promedio:            {kpi2:.2f} min")
print(f"KPI 3 — Distancia Asignación:        {kpi3:.2f} km")
print(f"KPI 4 — Espera en Restaurante:       {kpi4:.2f} min")
print(f"KPI 5 — Tiempo Total de Entrega:     {kpi5:.2f} min")

print(f"\nDistribución estados:")
print(df_gold['estado_actual'].value_counts())

print(f"\nCumplimiento SLA por franja:")
print(df_gold.groupby('franja_horaria')['cumple_sla'].mean().mul(100).round(2))

KPIs FINALES — CAPA GOLD

KPI 1 — Cumplimiento SLA:           92.42%
KPI 2 — Retraso Promedio:            0.27 min
KPI 3 — Distancia Asignación:        1.43 km
KPI 4 — Espera en Restaurante:       9.61 min
KPI 5 — Tiempo Total de Entrega:     17.66 min

Distribución estados:
estado_actual
Entregado                5452
Entregado con retraso    2524
Cancelado cliente         780
Cancelado sistema         748
Incidencia restaurant     496
Name: count, dtype: int64

Cumplimiento SLA por franja:
franja_horaria
Madrugada    100.00
Mañana       100.00
Noche         85.08
Tarde         84.83
Name: cumple_sla, dtype: float64


In [24]:
# CELDA 9 --- Exportar Gold

import os
import shutil

print("=" * 60)
print("ELIMINANDO ARCHIVOS ANTIGUOS")
print("=" * 60)

# Ruta exacta a tu carpeta data
ruta = '../../data/'

# Archivos antiguos a eliminar
archivos_antiguos = [
    'bronze_pedidos_raw.csv',
    'gold_dim_repartidor.csv', 
    'gold_fact_pedidos.csv',
    'silver_pedidos_limpios.csv'
]

for archivo in archivos_antiguos:
    ruta_completa = os.path.join(ruta, archivo)
    if os.path.exists(ruta_completa):
        os.remove(ruta_completa)
        print(f"Eliminado: {archivo}")
    else:
        print(f"No encontrado: {archivo}")

print("\n" + "=" * 60)
print("EXPORTANDO NUEVOS ARCHIVOS GOLD")
print("=" * 60)

# Tabla principal Gold
df_gold.to_csv(f'{ruta}gold_fact_pedidos_completo.csv', index=False)
print(f"gold_fact_pedidos_completo.csv → {len(df_gold):,} registros")

# Dimensiones limpias
df_dim_tiempo_s.to_csv(f'{ruta}gold_dim_tiempo.csv', index=False)
print(f"gold_dim_tiempo.csv → {len(df_dim_tiempo_s):,} registros")

df_dim_zona_s.to_csv(f'{ruta}gold_dim_zona.csv', index=False)
print(f"gold_dim_zona.csv → {len(df_dim_zona_s):,} registros")

df_dim_restaurante_s.to_csv(f'{ruta}gold_dim_restaurante.csv', index=False)
print(f"gold_dim_restaurante.csv → {len(df_dim_restaurante_s):,} registros")

df_dim_repartidor_s.to_csv(f'{ruta}gold_dim_repartidor.csv', index=False)
print(f"gold_dim_repartidor.csv → {len(df_dim_repartidor_s):,} registros")

df_dim_estado_pedido_s.to_csv(f'{ruta}gold_dim_estado_pedido.csv', index=False)
print(f"gold_dim_estado_pedido.csv → {len(df_dim_estado_pedido_s):,} registros")

df_dim_categoria_comida_s.to_csv(f'{ruta}gold_dim_categoria_comida.csv', index=False)
print(f"gold_dim_categoria_comida.csv → {len(df_dim_categoria_comida_s):,} registros")

df_dim_tipo_vehiculo_s.to_csv(f'{ruta}gold_dim_tipo_vehiculo.csv', index=False)
print(f"gold_dim_tipo_vehiculo.csv → {len(df_dim_tipo_vehiculo_s):,} registros")

df_dim_cliente_s.to_csv(f'{ruta}gold_dim_cliente.csv', index=False)
print(f"gold_dim_cliente.csv → {len(df_dim_cliente_s):,} registros")

df_dim_metodo_pago_s.to_csv(f'{ruta}gold_dim_metodo_pago.csv', index=False)
print(f"gold_dim_metodo_pago.csv → {len(df_dim_metodo_pago_s):,} registros")

df_dim_tipo_suscripcion_s.to_csv(f'{ruta}gold_dim_tipo_suscripcion.csv', index=False)
print(f"gold_dim_tipo_suscripcion.csv → {len(df_dim_tipo_suscripcion_s):,} registros")

df_dim_cadena_restaurante_s.to_csv(f'{ruta}gold_dim_cadena_restaurante.csv', index=False)
print(f"gold_dim_cadena_restaurante.csv → {len(df_dim_cadena_restaurante_s):,} registros")

print(f"\nTotal archivos nuevos: 12")
print(f"Guardados en: {os.path.abspath(ruta)}")

# Verificar contenido final de la carpeta data
print(f"\nContenido final de data/:")
for archivo in sorted(os.listdir(ruta)):
    if archivo.endswith('.csv'):
        tamaño = os.path.getsize(os.path.join(ruta, archivo))
        print(f"   {archivo} ({tamaño/1024:.1f} KB)")

ELIMINANDO ARCHIVOS ANTIGUOS
No encontrado: bronze_pedidos_raw.csv
Eliminado: gold_dim_repartidor.csv
No encontrado: gold_fact_pedidos.csv
No encontrado: silver_pedidos_limpios.csv

EXPORTANDO NUEVOS ARCHIVOS GOLD
gold_fact_pedidos_completo.csv → 10,000 registros
gold_dim_tiempo.csv → 5,988 registros
gold_dim_zona.csv → 5 registros
gold_dim_restaurante.csv → 10 registros
gold_dim_repartidor.csv → 200 registros
gold_dim_estado_pedido.csv → 5 registros
gold_dim_categoria_comida.csv → 5 registros
gold_dim_tipo_vehiculo.csv → 3 registros
gold_dim_cliente.csv → 3,000 registros
gold_dim_metodo_pago.csv → 5 registros
gold_dim_tipo_suscripcion.csv → 3 registros
gold_dim_cadena_restaurante.csv → 10 registros

Total archivos nuevos: 12
Guardados en: c:\PROJECT_RAPPI\data

Contenido final de data/:
   gold_dim_cadena_restaurante.csv (0.2 KB)
   gold_dim_categoria_comida.csv (0.2 KB)
   gold_dim_cliente.csv (52.5 KB)
   gold_dim_estado_pedido.csv (0.4 KB)
   gold_dim_metodo_pago.csv (0.2 KB)
   go